In [1]:
import pandas as pd                     # Data analysis and DataFrame operations
from geopy.geocoders import ArcGIS      # Geocoding addresses to obtain geographic coordinates
import time                             # Time-related utilities (used for request delays)
import folium                           # Interactive map generation and visualization
import osmnx as ox                      # OpenStreetMap road network extraction and analysis
import networkx as nx                   # Graph creation and shortest-path algorithms
import pyproj                           # Performs cartographic projections and geodetic/ellipsoidal calculations
from shapely.geometry import Point      # Geometric point representation for spatial analysis
import webbrowser                       # Opens generated map files in the default browser
import os                               # Operating system and file path management

In [2]:
data=pd.read_excel('addressList.xlsx')        # Read address data from Excel file
data                                          # display dataframe 

,Store Name,Address,Locality/Area,City,State,Pin Code,Landmark/Details
0,Vi Store Fort,"No 23B, Sir PM Road, Fort",Fort,Mumbai,Maharashtra,400001,Next to Hero Music House
1,R and R Enterprises (Vi Mini Store),"Shop No 19, Ground Floor, Plot No 66/74, Tabel...",Chira Bazar,Mumbai,Maharashtra,400002,Chandan Wadi
2,Shravani Enterprises (Vi Mini Store),"Shop No 3, 201, 211 Kesar Building, Princess S...",Marine Lines,Mumbai,Maharashtra,400002,Opp Geeta Bhuvan Hotel
3,GP Tel (Vi Mini Store),"Shop No 2, No 156/157, Panju Manzil, IR Road",IR Road,Mumbai,Maharashtra,400003,Opposite Metro Optician
4,Angel Telecom (Vi Mini Store),"S 325/A, Plot 323/325, Samual Street, Vadgadi,...",Masjid Bunder,Mumbai,Maharashtra,400003,Opposite Murshid Mansion
5,RK Enterprises (Vi Mini Store),"Shop No 2/156/158, Laxmi Niwas, CP Tank Road",CP Tank Road,Mumbai,Maharashtra,400004,Opposite Madhav Baug
6,Vi Store Parel,"Shop No.6, Kavarana Building, Dr. B A Road, Parel",Parel,Mumbai,Maharashtra,400012,Shri Sai Co
7,Vi Store Kurla East,"Shop 02 Usman Gani, Pawar House, Kurla East",Kurla East,Mumbai,Maharashtra,400024,Bhabha Hospital
8,Vi Store Santacruz West,"Orchid Pride, G 101, S V Road, Santacruz West",Santacruz West,Mumbai,Maharashtra,400054,"Near Corner Of Convent Avenue, Next To St Ther..."
9,Vi Store Borivali West,"Raichuria House, LT Cross Road, Borivali West",Borivali West,Mumbai,Maharashtra,400091,"Near Raichuria Circle, Opposite Vrundas Hotel"


In [3]:
def geocode_addresses(df):
    """
    Geocode addresses and return a new DataFrame containing
    Address, Latitude, and Longitude.

    Parameters:
        df (pd.DataFrame): Input DataFrame
        address_column (str): Name of address column

    Returns:
        pd.DataFrame: New DataFrame with coordinates
    """
    # Initialize ArcGIS geocoder with a 20-second timeout
    geolocator = ArcGIS(timeout=20)

    # List to store geocoding results
    result_data = []
    
    # Process each address in the DataFrame
    for _, row in df.iterrows():
        
        # Construct complete address string for geocoding
        address = f"{row['Address']}, {row['City']}, {row['State']}, India"

        try:
            # Retrieve geographic coordinates for the address
            location = geolocator.geocode(address)

            if location:
                # Store address and coordinates if geocoding succeeds
                result_data.append([
                    row['Address'],
                    location.latitude,
                    location.longitude
                ])
            else:
                # Store null values if no location is found
                result_data.append([
                    row['Address'],
                    None,
                    None
                ])

            # Delay requests to avoid exceeding service limits
            time.sleep(1)

        except Exception as e:
            # Store null values and log the error if geocoding fails
            result_data.append([
                row['Address'],
                None,
                None
            ])
            print(f"Error for address: {address}")
            print(e)
    
    # Convert results into a DataFrame and return
    return pd.DataFrame(
        result_data,
        columns=["Address", "Latitude", "Longitude"]
    )

In [4]:
new_data1=geocode_addresses(data)   # Geocode all addresses and obtain their latitude and longitude coordinates
new_data1                           # Display the resulting DataFrame containing geocoded locations

,Address,Latitude,Longitude
0,"No 23B, Sir PM Road, Fort",18.933570,72.838670
1,"Shop No 19, Ground Floor, Plot No 66/74, Tabel...",18.946044,72.827683
2,"Shop No 3, 201, 211 Kesar Building, Princess S...",18.945088,72.826718
3,"Shop No 2, No 156/157, Panju Manzil, IR Road",18.969047,72.821181
4,"S 325/A, Plot 323/325, Samual Street, Vadgadi,...",18.951011,72.836021
5,"Shop No 2/156/158, Laxmi Niwas, CP Tank Road",19.004773,72.858867
6,"Shop No.6, Kavarana Building, Dr. B A Road, Parel",19.004975,72.839584
7,"Shop 02 Usman Gani, Pawar House, Kurla East",19.170749,72.864853
8,"Orchid Pride, G 101, S V Road, Santacruz West",19.076054,72.837972
9,"Raichuria House, LT Cross Road, Borivali West",19.233776,72.856938


In [5]:
def get_coordinates(df):
    """
Geocodes addresses using address details and pincode information.

Parameters:
    df (pd.DataFrame): Input DataFrame containing Address, City, State, and Pin Code columns.

Returns:
    pd.DataFrame: DataFrame containing Address, Pincode, Latitude_Pincode, and Longitude_Pincode.
"""
    # List to store geocoding results
    result_data=[]

    # Initialize ArcGIS geocoder with a 20-second timeout
    geolocator = ArcGIS(timeout=20)

    # Process each record in the DataFrame
    for _, row in df.iterrows():

        # Construct full address including pincode
        address= f"{row['Address']}, {row['City']}, {row['State']}, {row['Pin Code']}, India"

        try:
            # Retrieve geographic coordinates for the address
            location = geolocator.geocode(address)

            if location:
                # Store address, pincode, and coordinates if geocoding succeeds
                result_data.append([
                    row['Address'],
                    row['Pin Code'],
                    location.latitude,
                    location.longitude
                ])
            else:
                # Store null values if no location is found
                result_data.append([
                    row['Address'],
                    row['Pin Code'],
                    None,
                    None
                ])

            time.sleep(1)

        except Exception as e:
            result_data.append([
                row['Address'],
                row['Pin Code'],
                None,
                None
            ])
            print(f"Error for address: {address}")
            print(e)

    return pd.DataFrame(
        result_data,
        columns=["Address","Pincode","Latitude_Pincode", "Longitude_Pincode"]
    )

In [6]:
new_data2=get_coordinates(data)          # Geocode addresses using address and pincode information
new_data2                                # Display the resulting coordinates obtained from pincode-based geocoding

,Address,Pincode,Latitude_Pincode,Longitude_Pincode
0,"No 23B, Sir PM Road, Fort",400001,18.941845,72.838958
1,"Shop No 19, Ground Floor, Plot No 66/74, Tabel...",400002,18.946044,72.827683
2,"Shop No 3, 201, 211 Kesar Building, Princess S...",400002,18.945088,72.826718
3,"Shop No 2, No 156/157, Panju Manzil, IR Road",400003,18.950423,72.837882
4,"S 325/A, Plot 323/325, Samual Street, Vadgadi,...",400003,18.951011,72.836021
5,"Shop No 2/156/158, Laxmi Niwas, CP Tank Road",400004,18.955389,72.826213
6,"Shop No.6, Kavarana Building, Dr. B A Road, Parel",400012,19.004975,72.839584
7,"Shop 02 Usman Gani, Pawar House, Kurla East",400024,19.170749,72.864853
8,"Orchid Pride, G 101, S V Road, Santacruz West",400054,19.076054,72.837972
9,"Raichuria House, LT Cross Road, Borivali West",400091,19.231321,72.841312


In [7]:
# Merge address-based and pincode-based geocoding results
new_data=pd.merge(new_data1,new_data2,on=['Address'],how='left')

# Add store names to the merged dataset
new_data['Store Name']=data['Store Name']

# Calculate the absolute difference in latitude values
new_data['Delta Latitude']=abs(new_data['Latitude']-new_data['Latitude_Pincode'])


# Calculate the absolute difference in longitude values
new_data['Delta Longitude']=abs(new_data['Longitude']-new_data['Longitude_Pincode'])

# Display the combined dataset and coordinate differences
new_data

,Address,Latitude,Longitude,Pincode,Latitude_Pincode,Longitude_Pincode,Store Name,Delta Latitude,Delta Longitude
0,"No 23B, Sir PM Road, Fort",18.933570,72.838670,400001,18.941845,72.838958,Vi Store Fort,0.008275,0.000288
1,"Shop No 19, Ground Floor, Plot No 66/74, Tabel...",18.946044,72.827683,400002,18.946044,72.827683,R and R Enterprises (Vi Mini Store),0.000000,0.000000
2,"Shop No 3, 201, 211 Kesar Building, Princess S...",18.945088,72.826718,400002,18.945088,72.826718,Shravani Enterprises (Vi Mini Store),0.000000,0.000000
3,"Shop No 2, No 156/157, Panju Manzil, IR Road",18.969047,72.821181,400003,18.950423,72.837882,GP Tel (Vi Mini Store),0.018624,0.016702
4,"S 325/A, Plot 323/325, Samual Street, Vadgadi,...",18.951011,72.836021,400003,18.951011,72.836021,Angel Telecom (Vi Mini Store),0.000000,0.000000
5,"Shop No 2/156/158, Laxmi Niwas, CP Tank Road",19.004773,72.858867,400004,18.955389,72.826213,RK Enterprises (Vi Mini Store),0.049384,0.032654
6,"Shop No.6, Kavarana Building, Dr. B A Road, Parel",19.004975,72.839584,400012,19.004975,72.839584,Vi Store Parel,0.000000,0.000000
7,"Shop 02 Usman Gani, Pawar House, Kurla East",19.170749,72.864853,400024,19.170749,72.864853,Vi Store Kurla East,0.000000,0.000000
8,"Orchid Pride, G 101, S V Road, Santacruz West",19.076054,72.837972,400054,19.076054,72.837972,Vi Store Santacruz West,0.000000,0.000000
9,"Raichuria House, LT Cross Road, Borivali West",19.233776,72.856938,400091,19.231321,72.841312,Vi Store Borivali West,0.002455,0.015626


In [8]:
def calculate_geodesic_distance(lat1, lon1, lat2, lon2):
    """
    Calculates the high-precision geodesic distance between two points 
    on the WGS84 ellipsoidal surface using pyproj.
    """
    # Load standard WGS84 ellipsoid
    geod = pyproj.Geod(ellps="WGS84")
    
    # pyproj expects coordinates in (Longitude, Latitude) order
    _, _, dist_meters = geod.inv(lon1, lat1, lon2, lat2)
    
    # Convert meters to kilometers
    return dist_meters / 1000.0

In [9]:
def calculate_shortest_road_path(start_coords, end_coords,graph,graph_proj):

    # Extract latitude and longitude from the start and end coordinates
    lat1, lon1 = start_coords[0], start_coords[1]
    lat2, lon2 = end_coords[0], end_coords[1]  

    # Calculate the straight-line (geodesic) distance as a fallback
    geodesic_dis=calculate_geodesic_distance(lat1, lon1, lat2, lon2)

    try:
    # Create Shapely Point objects using (longitude, latitude)
     point1 = Point(lon1, lat1)
     point2 = Point(lon2, lat2)
    
    # Project points into the same coordinate reference system (CRS) as the projected road network graph
     point1_proj, _ = ox.projection.project_geometry(point1, crs=graph.graph['crs'], to_crs=graph_proj.graph['crs'])
     point2_proj, _ = ox.projection.project_geometry(point2, crs=graph.graph['crs'], to_crs=graph_proj.graph['crs'])
    
    # Find the nearest road network nodes to the projected points
     start_node = ox.nearest_nodes(graph_proj, X=point1_proj.x, Y=point1_proj.y) 
     end_node = ox.nearest_nodes(graph_proj, X=point2_proj.x, Y=point2_proj.y)

    # Compute the shortest road path based on edge length
     shortest_route = nx.shortest_path(graph_proj, start_node, end_node, weight="length") 
    
    # Convert the route into a GeoDataFrame and calculate total road distance
     gdf_edges = ox.routing.route_to_gdf(graph_proj, shortest_route)
     actual_road_km = gdf_edges["length"].sum() / 1000 
    
    # Extract route coordinates for map visualization
     route_coordinates = []
     for node in shortest_route:
        node_data = graph.nodes[node]
        route_coordinates.append((node_data['y'], node_data['x']))

    # Return road distance and route coordinates    
     return actual_road_km, route_coordinates
    
    except nx.NetworkXNoPath:
        # Handle cases where no road connection exists between locations
        print("No road path exists between these locations, using geodesic distance.")
        return geodesic_dis,[]
    
    except Exception as e:
        # Handle routing or projection errors and fall back to geodesic distance
        print(f"Road routing unavailable: {e}")
        # Fallback to geodesic distance
        return geodesic_dis,[]


In [28]:
def process_map1(df,graph,graph_proj):
    # Initialize total trip distance accumulator
    total_trip_distance = 0.0
    
    lat=(df['Latitude'].max()+df['Latitude'].min())/2
    lon=(df['Longitude'].max()+df['Longitude'].min())/2
    google_tiles_url = "https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}" 
    m = folium.Map(location=[lat,lon], 
                   zoom_start=11,
                   tiles=google_tiles_url, 
                   attr='Google'
                  )
    blue_group = folium.FeatureGroup(name="Same Coordinates")
    red_group = folium.FeatureGroup(name="Address Only")
    green_group = folium.FeatureGroup(name="Address + Pincode")

    for _,row in df.iterrows():
        popup_text = (
        f"Store Name: {row['Store Name']}<br>"
        f"Latitude: {lat}<br>"
        f"Longitude: {lon}"
    )
        if (row['Delta Latitude']!=0 or row['Delta Longitude']!=0):
            folium.Marker(location=[row['Latitude'],row['Longitude']],
                        popup=folium.Popup(popup_text, max_width=250),
                        icon=folium.Icon(color='red')
                         ).add_to(red_group)
            folium.Marker(location=[row['Latitude_Pincode'],row['Longitude_Pincode']],
                        popup=folium.Popup(popup_text, max_width=250),
                        icon=folium.Icon(color='green')
                         ).add_to(green_group)
        else:
           folium.Marker(location=[row['Latitude'],row['Longitude']],
                        popup=folium.Popup(popup_text, max_width=250),
                        icon=folium.Icon(color='blue')
                         ).add_to(blue_group) 
    blue_group.add_to(m)
    red_group.add_to(m)
    green_group.add_to(m)

    folium.LayerControl().add_to(m)
    # Iterate through consecutive pairs of locations
    for i in range(len(df) - 1):

        # Current point and next point coordinates
        start_pt = (df.loc[i, 'Latitude'], df.loc[i, 'Longitude'])
        end_pt = (df.loc[i+1, 'Latitude'], df.loc[i+1, 'Longitude'])

        # Calculate shortest road path and distance between points
        segment_km, route_coords = calculate_shortest_road_path(start_pt, end_pt,graph,graph_proj)

        # Print whether road-network distance or geodesic distance was used
        if route_coords and len(route_coords) > 0:
            print(f"Actual distance between Store {i+1} and Store {i+2} is:{segment_km:.2f} km")
        else:
            print(f"Geodesic distance between Store {i+1} and Store {i+2} is:{segment_km:.2f} km")
        
        # Add segment distance to total trip distance
        total_trip_distance += segment_km
        # Draw route on map
        if route_coords and len(route_coords) > 0:
         # Draw actual road path if route coordinates are available
         folium.PolyLine(
            locations=route_coords, 
            color="blue", 
            weight=5, 
            opacity=0.8,
            popup=f"Leg {i+1}: {segment_km:.2f} km"
         ).add_to(m)
        else:
         # Draw straight-line path when road route cannot be found
         folium.PolyLine(
            locations=[start_pt, end_pt],
            color="gray",
            weight=3,
            opacity=0.6,
            dash_array="5,5",
            popup=f"Leg {i+1}: {segment_km:.2f} km (geodesic)"
         ).add_to(m)

    # Display total cumulative distance of the trip
    print(f"Total cumulative road distance across all stops when locations are pointed using only address: {total_trip_distance:.2f} km")
    # Save map as an HTML file
    file_name = "add_mapping1.html"
    m.save(file_name)  

    # Open the generated map in the default web browser             
    webbrowser.open('file://' + os.path.realpath(file_name)) 

In [31]:
def process_map2(df,graph,graph_proj):
    total_trip_distance = 0.0
    
    lat=(df['Latitude'].max()+df['Latitude'].min())/2
    lon=(df['Longitude'].max()+df['Longitude'].min())/2
    google_tiles_url = "https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}" 
    m = folium.Map(location=[lat,lon], 
                   zoom_start=11,
                   tiles=google_tiles_url, 
                   attr='Google'
                  )
    blue_group = folium.FeatureGroup(name="Same Coordinates")
    red_group = folium.FeatureGroup(name="Address Only")
    green_group = folium.FeatureGroup(name="Address + Pincode")
    # Iterate through consecutive pairs of locations
    for _,row in df.iterrows():
        popup_text = (
        f"Store Name: {row['Store Name']}<br>"
        f"Latitude: {lat}<br>"
        f"Longitude: {lon}"
        )
        if (row['Delta Latitude']!=0 or row['Delta Longitude']!=0):
            folium.Marker(location=[row['Latitude'],row['Longitude']],
                        popup=folium.Popup(popup_text, max_width=250),
                        icon=folium.Icon(color='red')
                         ).add_to(red_group)
            folium.Marker(location=[row['Latitude_Pincode'],row['Longitude_Pincode']],
                        popup=folium.Popup(popup_text, max_width=250),
                        icon=folium.Icon(color='green')
                         ).add_to(green_group)
        else:
           folium.Marker(location=[row['Latitude'],row['Longitude']],
                        popup=folium.Popup(popup_text, max_width=250),
                        icon=folium.Icon(color='blue')
                         ).add_to(blue_group)
    blue_group.add_to(m)
    red_group.add_to(m)
    green_group.add_to(m)

    folium.LayerControl().add_to(m) 
    for i in range(len(df) - 1):

        # Current point and next point coordinates
        start_pt = (df.loc[i, 'Latitude_Pincode'], df.loc[i, 'Longitude_Pincode'])
        end_pt = (df.loc[i+1, 'Latitude_Pincode'], df.loc[i+1, 'Longitude_Pincode'])

        # Calculate shortest road path and distance between points
        segment_km, route_coords = calculate_shortest_road_path(start_pt, end_pt,graph,graph_proj)

        # Print whether road-network distance or geodesic distance was used
        if route_coords and len(route_coords) > 0:
            print(f"Actual distance between Store {i+1} and Store {i+2} is:{segment_km:.2f} km")
        else:
            print(f"Geodesic distance between Store {i+1} and Store {i+2} is:{segment_km:.2f} km")
        
        # Add segment distance to total trip distance
        total_trip_distance += segment_km
        # Draw route on map
        if route_coords and len(route_coords) > 0:
         # Draw actual road path if route coordinates are available
         folium.PolyLine(
            locations=route_coords, 
            color="darkgreen", 
            weight=5, 
            opacity=0.8,
            popup=f"Leg {i+1}: {segment_km:.2f} km"
         ).add_to(m)
        else:
         # Draw straight-line path when road route cannot be found
         folium.PolyLine(
            locations=[start_pt, end_pt],
            color="gray",
            weight=3,
            opacity=0.6,
            dash_array="5,5",
            popup=f"Leg {i+1}: {segment_km:.2f} km (geodesic)"
         ).add_to(m)

    # Display total cumulative distance of the trip
    print(f"Total cumulative road distance across all stops when locations are pointed using address and pincode: {total_trip_distance:.2f} km")
    file_name = "add_mapping2.html"
    m.save(file_name)  

    # Open the generated map in the default web browser             
    webbrowser.open('file://' + os.path.realpath(file_name)) 


In [32]:
padding = 0.03                                                 

# Determine geographic bounding box limits
north =new_data["Latitude"].max() + padding
south =new_data["Latitude"].min() - padding
east =new_data["Longitude"].max() + padding
west =new_data["Longitude"].min() - padding 
    
# Download drivable road network within the bounding box
graph = ox.graph_from_bbox(bbox=(west, south, east, north), network_type="drive")

# Verify that a valid road network was retrieved
if len(graph) == 0:
        raise ValueError("No drivable roads found in this specific coordinate region.")
    
# Project graph to a suitable coordinate reference system for accurate distance calculations
graph_proj = ox.project_graph(graph)
    
# Calculate distances, generate route map,and display/save the final output
process_map1(new_data,graph,graph_proj)
process_map2(new_data,graph,graph_proj)

Actual distance between Store 1 and Store 2 is:2.07 km
Actual distance between Store 2 and Store 3 is:0.79 km
Actual distance between Store 3 and Store 4 is:3.45 km
Actual distance between Store 4 and Store 5 is:3.12 km
Actual distance between Store 5 and Store 6 is:7.27 km
Actual distance between Store 6 and Store 7 is:3.43 km
Actual distance between Store 7 and Store 8 is:21.13 km
Actual distance between Store 8 and Store 9 is:12.06 km
Actual distance between Store 9 and Store 10 is:18.47 km
Actual distance between Store 10 and Store 11 is:21.40 km
Actual distance between Store 11 and Store 12 is:9.78 km
Actual distance between Store 12 and Store 13 is:5.54 km
Actual distance between Store 13 and Store 14 is:3.68 km
Actual distance between Store 14 and Store 15 is:13.37 km
Actual distance between Store 15 and Store 16 is:5.90 km
Actual distance between Store 16 and Store 17 is:11.81 km
Actual distance between Store 17 and Store 18 is:18.05 km
Actual distance between Store 18 and Stor

In [33]:
print(new_data['Delta Latitude'].max(),new_data['Delta Longitude'].max())

0.0493837008480007 0.032654302208996455
